---
# Config B — Deep RL on the Clinical ICU-Sepsis Environment

In Config A, the patient state was a single discrete integer (0–715), making tabular methods feasible. Config B introduces two fundamental changes that make tabular methods impossible:

1. **Continuous 47-dimensional observations** — the agent receives the actual normalised physiological measurements (SOFA score, heart rate, lactate, blood pressure, creatinine, and 42 others). With continuous observations, the agent never sees the exact same state twice, so a Q-table with one entry per state is no longer viable.

2. **Clinical reality wrappers** — three failure modes are injected that reflect real ICU deployment challenges: episodic sensor noise, episodic missing lab values, and rare acute patient deterioration events.

The solution is **function approximation** — replacing the Q-table with a neural network that generalises across similar states. This is the core motivation for Deep RL.

We implement two algorithms:
- **DQN** (Deep Q-Network): the direct deep learning extension of Q-Learning from Config A. Same off-policy principle, neural network replaces the Q-table.
- **PPO** (Proximal Policy Optimisation): a policy-based deep RL algorithm. Directly learns a probability distribution over actions. On-policy like SARSA was in Config A.

This preserves the **off-policy vs on-policy contrast** from Config A at the deep RL level, making the cross-config comparison meaningful.


---
## 8. Observation Space Analysis

Before training any agent, we explore the 47-dimensional observation space. This matters because:
- It tells us which features vary most across patients (most informative for learning)
- It shows which features get zeroed out by the missing wrapper
- It shows how much noise the noisy wrapper adds

Understanding what the agent actually sees is essential for interpreting its behaviour clinically.


In [ ]:
# ── Observation space analysis ───────────────────────────────────────────────
# Collect observations from 500 random episodes across all wrapper types

np.random.seed(SEED)
env_obs = make_clinical_env()

all_obs        = []
noisy_obs      = []
clean_obs      = []
missing_obs    = []
nonmissing_obs = []

N_ANALYSIS_EPISODES = 500

for ep in range(N_ANALYSIS_EPISODES):
    obs, info = env_obs.reset(seed=np.random.randint(100_000))
    ep_noisy   = info.get('noisy_episode', False)
    ep_missing = info.get('missing_features') is not None
    done = False

    while not done:
        all_obs.append(obs.copy())
        if ep_noisy:   noisy_obs.append(obs.copy())
        else:          clean_obs.append(obs.copy())
        if ep_missing: missing_obs.append(obs.copy())
        else:          nonmissing_obs.append(obs.copy())

        obs, _, te, tr, info = env_obs.step(env_obs.action_space.sample())
        done = te or tr

env_obs.close()

all_obs        = np.array(all_obs)
clean_obs      = np.array(clean_obs)
noisy_obs      = np.array(noisy_obs)     if noisy_obs      else np.zeros((1, 47))
missing_obs    = np.array(missing_obs)   if missing_obs    else np.zeros((1, 47))
nonmissing_obs = np.array(nonmissing_obs)

print(f'Total observations collected : {len(all_obs):,}')
print(f'Clean episode observations   : {len(clean_obs):,}')
print(f'Noisy episode observations   : {len(noisy_obs):,}')
print(f'Missing episode observations : {len(missing_obs):,}')
print(f'Observation vector shape     : {all_obs.shape}')
print(f'Feature value range          : [{all_obs.min():.3f}, {all_obs.max():.3f}]')


In [ ]:
# ── Plot: observation space exploration ───────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: feature variance across all episodes (which features vary most?)
feature_std = all_obs.std(axis=0)
top10_idx   = np.argsort(feature_std)[-10:][::-1]
colors_std  = ['tomato' if i in top10_idx else 'steelblue' for i in range(47)]

axes[0].bar(range(47), feature_std, color=colors_std, alpha=0.8, width=0.8)
axes[0].set_xlabel('Feature index')
axes[0].set_ylabel('Standard deviation across observations')
axes[0].set_title('Feature Variability\n(red = top 10 most variable)', fontweight='bold')

# Middle: effect of missing wrapper — how many features become zero?
if len(missing_obs) > 1:
    missing_zero_rate    = (missing_obs    == 0).mean(axis=0)
    nonmissing_zero_rate = (nonmissing_obs == 0).mean(axis=0)
    axes[1].bar(range(47), missing_zero_rate,
                color='tomato', alpha=0.7, label='Missing episode', width=0.8)
    axes[1].bar(range(47), nonmissing_zero_rate,
                color='steelblue', alpha=0.7, label='Normal episode', width=0.8)
    axes[1].set_xlabel('Feature index')
    axes[1].set_ylabel('Proportion zeroed out')
    axes[1].set_title('Missing Wrapper: Feature Zero Rate\n(which features go missing?)',
                      fontweight='bold')
    axes[1].legend(fontsize=8)
else:
    axes[1].text(0.5, 0.5, 'No missing episodes\nin sample',
                 ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title('Missing Wrapper: Feature Zero Rate', fontweight='bold')

# Right: effect of noise wrapper — distribution shift per feature
if len(noisy_obs) > 1 and len(clean_obs) > 1:
    noise_delta = np.abs(noisy_obs.mean(axis=0) - clean_obs.mean(axis=0))
    axes[2].bar(range(47), noise_delta,
                color='darkorange', alpha=0.8, width=0.8)
    axes[2].set_xlabel('Feature index')
    axes[2].set_ylabel('|Mean shift| due to noise')
    axes[2].set_title('Noisy Wrapper: Mean Feature Shift\n(how much does noise distort each feature?)',
                      fontweight='bold')
else:
    axes[2].text(0.5, 0.5, 'No noisy episodes\nin sample',
                 ha='center', va='center', transform=axes[2].transAxes)
    axes[2].set_title('Noisy Wrapper: Mean Feature Shift', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configB_obs_analysis.png', bbox_inches='tight')
plt.show()

print(f'Top 10 most variable features: {sorted(top10_idx.tolist())}')
print(f'Mean feature std across all observations: {feature_std.mean():.4f}')


---
## 9. DQN — Deep Q-Network

DQN (Mnih et al., 2015) replaces the Q-table from Config A with a neural network. Given the 47-dimensional observation, the network outputs 25 Q-values — one per action. The agent picks the action with the highest Q-value (greedy) or a random action (exploration).

Two key stabilisation tricks distinguish DQN from naive deep Q-Learning:

**Replay buffer**: experiences `(s, a, r, s')` are stored in a memory buffer. The network trains on random batches sampled from this buffer, breaking the temporal correlation between consecutive updates that would otherwise destabilise training.

**Target network**: a separate frozen copy of the network is used to compute the TD target `r + γ * max Q(s')`. This target network is updated every few hundred steps, preventing the "moving target" problem where the network chases its own changing predictions.

The connection to Config A is direct — DQN is Q-Learning with a neural network. The off-policy, greedy-target principle is identical.


In [ ]:
# ── Install and import stable-baselines3 ─────────────────────────────────────
# !pip install stable-baselines3 torch

from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
from stable_baselines3.common.monitor import Monitor
import torch

print(f'stable-baselines3 imported successfully')
print(f'PyTorch version : {torch.__version__}')


In [ ]:
# ── Training callback: log episode returns ────────────────────────────────────

class ReturnLoggerCallback(BaseCallback):
    """
    Logs mean episode return every eval_freq timesteps.
    Stores results in self.return_log for plotting.
    """
    def __init__(self, eval_env, eval_freq=2000, n_eval_episodes=200, verbose=0):
        super().__init__(verbose)
        self.eval_env       = eval_env
        self.eval_freq      = eval_freq
        self.n_eval_episodes = n_eval_episodes
        self.return_log     = []   # (timestep, mean_return, survival_rate)

    def _on_step(self):
        if self.n_calls % self.eval_freq == 0:
            returns = []
            for _ in range(self.n_eval_episodes):
                obs, _ = self.eval_env.reset(
                    seed=np.random.randint(100_000))
                total_r, done = 0.0, False
                while not done:
                    action, _ = self.model.predict(obs, deterministic=True)
                    obs, r, te, tr, _ = self.eval_env.step(int(action))
                    total_r += r; done = te or tr
                returns.append(total_r)
            returns = np.array(returns)
            self.return_log.append((
                self.n_calls,
                float(np.mean(returns)),
                float(np.mean(returns > 0)) * 100,
            ))
            if self.verbose:
                print(f'  Step {self.n_calls:>8,} | '
                      f'mean return: {np.mean(returns):.4f} | '
                      f'survival: {np.mean(returns > 0)*100:.1f}%')
        return True


In [ ]:
# ── DQN training ─────────────────────────────────────────────────────────────
# Hyperparameters follow established defaults for discrete-action clinical RL
# learning_rate=1e-4: standard for DQN, conservative enough for sparse rewards
# buffer_size=50k: large enough to decorrelate experiences
# batch_size=64: standard minibatch size
# learning_starts=1000: fill buffer before training begins
# target_update_interval=500: update target network every 500 steps
# exploration_fraction=0.3: spend 30% of training exploring
# gamma=1.0: consistent with Config A (no discounting, episodic task)

N_TIMESTEPS_DQN = 200_000

np.random.seed(SEED)
env_dqn_train = make_clinical_env()
env_dqn_eval  = make_clinical_env()

dqn_callback = ReturnLoggerCallback(
    eval_env=env_dqn_eval,
    eval_freq=2000,
    n_eval_episodes=200,
    verbose=1,
)

dqn_model = DQN(
    policy              = 'MlpPolicy',
    env                 = env_dqn_train,
    learning_rate       = 1e-4,
    buffer_size         = 50_000,
    learning_starts     = 1_000,
    batch_size          = 64,
    gamma               = 1.0,
    target_update_interval = 500,
    exploration_fraction   = 0.3,
    exploration_final_eps  = 0.05,
    verbose             = 0,
    seed                = SEED,
)

print(f'Training DQN | {N_TIMESTEPS_DQN:,} timesteps...')
dqn_model.learn(total_timesteps=N_TIMESTEPS_DQN, callback=dqn_callback)
print('DQN training complete.')

env_dqn_train.close()
env_dqn_eval.close()


In [ ]:
# ── Evaluate DQN ─────────────────────────────────────────────────────────────

def evaluate_deep_agent(model, n_episodes=1000, seed=SEED, deterministic=True):
    """
    Evaluate a stable-baselines3 model over n_episodes.
    Returns dict with mean_return, survival_rate, std_return, mean_length.
    """
    np.random.seed(seed)
    env_eval = make_clinical_env()
    returns, lengths = [], []

    for _ in range(n_episodes):
        obs, _ = env_eval.reset(seed=np.random.randint(100_000))
        total_r, steps, done = 0.0, 0, False
        while not done:
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, r, te, tr, _ = env_eval.step(int(action))
            total_r += r; steps += 1; done = te or tr
        returns.append(total_r)
        lengths.append(steps)

    env_eval.close()
    returns = np.array(returns)
    return {
        'mean_return'   : float(np.mean(returns)),
        'std_return'    : float(np.std(returns)),
        'survival_rate' : float(np.mean(returns > 0)) * 100,
        'mean_length'   : float(np.mean(lengths)),
        'returns_array' : returns,
    }


dqn_results = evaluate_deep_agent(dqn_model, n_episodes=1000)
print('DQN — Evaluation (1000 episodes):')
print(f'  Mean return   : {dqn_results["mean_return"]:.4f}')
print(f'  Survival rate : {dqn_results["survival_rate"]:.1f}%')
print(f'  Std return    : {dqn_results["std_return"]:.4f}')
print(f'  Mean length   : {dqn_results["mean_length"]:.1f} steps')
print(f'  vs Random baseline: {clinical_rand_mean:.4f}')


In [ ]:
# ── Plot DQN: learning curve ──────────────────────────────────────────────────

dqn_steps    = [x[0] for x in dqn_callback.return_log]
dqn_surv_log = [x[2] for x in dqn_callback.return_log]
dqn_ret_log  = [x[1] for x in dqn_callback.return_log]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(dqn_steps, dqn_surv_log, color='steelblue', linewidth=2.0, label='DQN')
axes[0].axhline(clinical_rand_mean * 100 if clinical_rand_mean < 1
                else clinical_rand_mean,
                color='gray', linestyle=':', linewidth=1.5, label=f'Random baseline')
axes[0].axhline(np.mean(np.array(clinical_rand_returns) > 0) * 100,
                color='gray', linestyle=':', linewidth=1.5)
axes[0].set_xlabel('Training timesteps')
axes[0].set_ylabel('Survival Rate (%)')
axes[0].set_title('DQN — Survival Rate During Training', fontweight='bold')
axes[0].legend(fontsize=8)

axes[1].plot(dqn_steps, dqn_ret_log, color='steelblue', linewidth=2.0, label='DQN')
axes[1].axhline(clinical_rand_mean, color='gray', linestyle=':',
                linewidth=1.5, label='Random baseline')
axes[1].set_xlabel('Training timesteps')
axes[1].set_ylabel('Mean return')
axes[1].set_title('DQN — Mean Return During Training', fontweight='bold')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configB_DQN_training.png', bbox_inches='tight')
plt.show()


---
## 10. PPO — Proximal Policy Optimisation

PPO (Schulman et al., 2017) is a policy-based deep RL algorithm. Instead of learning Q-values and deriving a policy from them (like DQN), PPO directly learns a probability distribution over actions — a stochastic policy π(a|s).

The "proximal" part refers to a clipping mechanism that prevents policy updates from being too large in a single step. Without this, policy gradient methods can catastrophically forget what they learned. PPO clips the update so the new policy never strays too far from the old one, making training much more stable.

Key differences from DQN in this context:
- **On-policy**: PPO learns from its own current behaviour, like SARSA did in Config A. It cannot reuse old experiences (no replay buffer).
- **Stochastic policy**: outputs action probabilities rather than Q-values. This can be more robust to noisy observations because the agent doesn't commit rigidly to one action.
- **Actor-Critic architecture**: maintains two networks — an actor (the policy) and a critic (a value function estimator). The critic helps reduce variance in the policy gradient updates.


In [ ]:
# ── PPO training ─────────────────────────────────────────────────────────────
# Hyperparameters follow Schulman et al. (2017) defaults adapted for episodic tasks
# learning_rate=3e-4: standard PPO default
# n_steps=512: number of steps per environment before each update
# batch_size=64: minibatch size for gradient updates
# n_epochs=10: number of passes over collected data per update
# gamma=1.0: consistent with Config A (no discounting)
# clip_range=0.2: standard PPO clipping parameter

from stable_baselines3 import PPO

N_TIMESTEPS_PPO = 200_000

np.random.seed(SEED)
env_ppo_train = make_clinical_env()
env_ppo_eval  = make_clinical_env()

ppo_callback = ReturnLoggerCallback(
    eval_env=env_ppo_eval,
    eval_freq=2000,
    n_eval_episodes=200,
    verbose=1,
)

ppo_model = PPO(
    policy      = 'MlpPolicy',
    env         = env_ppo_train,
    learning_rate = 3e-4,
    n_steps     = 512,
    batch_size  = 64,
    n_epochs    = 10,
    gamma       = 1.0,
    clip_range  = 0.2,
    verbose     = 0,
    seed        = SEED,
)

print(f'Training PPO | {N_TIMESTEPS_PPO:,} timesteps...')
ppo_model.learn(total_timesteps=N_TIMESTEPS_PPO, callback=ppo_callback)
print('PPO training complete.')

env_ppo_train.close()
env_ppo_eval.close()


In [ ]:
# ── Evaluate PPO ─────────────────────────────────────────────────────────────
ppo_results = evaluate_deep_agent(ppo_model, n_episodes=1000)
print('PPO — Evaluation (1000 episodes):')
print(f'  Mean return   : {ppo_results["mean_return"]:.4f}')
print(f'  Survival rate : {ppo_results["survival_rate"]:.1f}%')
print(f'  Std return    : {ppo_results["std_return"]:.4f}')
print(f'  Mean length   : {ppo_results["mean_length"]:.1f} steps')
print(f'  vs Random baseline: {clinical_rand_mean:.4f}')


In [ ]:
# ── Plot PPO + DQN vs PPO overlay ─────────────────────────────────────────────

ppo_steps    = [x[0] for x in ppo_callback.return_log]
ppo_surv_log = [x[2] for x in ppo_callback.return_log]
ppo_ret_log  = [x[1] for x in ppo_callback.return_log]

rand_survival_B = float(np.mean(np.array(clinical_rand_returns) > 0)) * 100
rand_return_B   = float(np.mean(clinical_rand_returns))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: PPO learning curve
axes[0].plot(ppo_steps, ppo_surv_log, color='darkorange', linewidth=2.0, label='PPO')
axes[0].axhline(rand_survival_B, color='gray', linestyle=':', linewidth=1.5,
                label=f'Random baseline ({rand_survival_B:.1f}%)')
axes[0].set_xlabel('Training timesteps')
axes[0].set_ylabel('Survival Rate (%)')
axes[0].set_title('PPO — Survival Rate During Training', fontweight='bold')
axes[0].legend(fontsize=8)

# Right: DQN vs PPO head to head
axes[1].plot(dqn_steps, dqn_surv_log, color='steelblue',  linewidth=2.0, label='DQN')
axes[1].plot(ppo_steps, ppo_surv_log, color='darkorange', linewidth=2.0, label='PPO')
axes[1].axhline(rand_survival_B, color='gray', linestyle=':', linewidth=1.5,
                label=f'Random baseline ({rand_survival_B:.1f}%)')
axes[1].set_xlabel('Training timesteps')
axes[1].set_ylabel('Survival Rate (%)')
axes[1].set_title('DQN vs PPO — Learning Curves', fontweight='bold')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configB_PPO_training.png', bbox_inches='tight')
plt.show()


---
## 11. Hyperparameter Sensitivity Check

Rather than an exhaustive grid search (which would take hours at 200k timesteps per run), we perform a focused sensitivity check on the most impactful hyperparameter for each algorithm: the **learning rate**.

We test two values per algorithm — our default and a more aggressive alternative — and compare final survival rate. This validates our default choices and shows the sensitivity of each algorithm to this parameter.

This approach is standard in deep RL practice: informed defaults from the literature are used as the baseline, and a small sensitivity check validates them rather than searching blindly.


In [ ]:
# ── Learning rate sensitivity: DQN ───────────────────────────────────────────
# Default: 1e-4 (already trained above)
# Alternative: 5e-4 (more aggressive, risks instability)

print('Training DQN | learning_rate=5e-4 | 200k timesteps...')
np.random.seed(SEED)
env_dqn_lr2 = make_clinical_env()

dqn_lr2 = DQN(
    policy='MlpPolicy', env=env_dqn_lr2,
    learning_rate=5e-4, buffer_size=50_000,
    learning_starts=1_000, batch_size=64,
    gamma=1.0, target_update_interval=500,
    exploration_fraction=0.3, exploration_final_eps=0.05,
    verbose=0, seed=SEED,
)
dqn_lr2_cb = ReturnLoggerCallback(
    eval_env=make_clinical_env(), eval_freq=2000,
    n_eval_episodes=200, verbose=0,
)
dqn_lr2.learn(total_timesteps=200_000, callback=dqn_lr2_cb)
env_dqn_lr2.close()

dqn_lr2_results = evaluate_deep_agent(dqn_lr2, n_episodes=1000)
print(f'  DQN lr=1e-4: {dqn_results["survival_rate"]:.1f}%  '
      f'(default)')
print(f'  DQN lr=5e-4: {dqn_lr2_results["survival_rate"]:.1f}%')


In [ ]:
# ── Learning rate sensitivity: PPO ───────────────────────────────────────────
# Default: 3e-4 (already trained above)
# Alternative: 1e-3 (more aggressive)

print('Training PPO | learning_rate=1e-3 | 200k timesteps...')
np.random.seed(SEED)
env_ppo_lr2 = make_clinical_env()

ppo_lr2 = PPO(
    policy='MlpPolicy', env=env_ppo_lr2,
    learning_rate=1e-3, n_steps=512,
    batch_size=64, n_epochs=10,
    gamma=1.0, clip_range=0.2,
    verbose=0, seed=SEED,
)
ppo_lr2_cb = ReturnLoggerCallback(
    eval_env=make_clinical_env(), eval_freq=2000,
    n_eval_episodes=200, verbose=0,
)
ppo_lr2.learn(total_timesteps=200_000, callback=ppo_lr2_cb)
env_ppo_lr2.close()

ppo_lr2_results = evaluate_deep_agent(ppo_lr2, n_episodes=1000)
print(f'  PPO lr=3e-4: {ppo_results["survival_rate"]:.1f}%  '
      f'(default)')
print(f'  PPO lr=1e-3: {ppo_lr2_results["survival_rate"]:.1f}%')


In [ ]:
# ── Plot: learning rate sensitivity ──────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# DQN: default vs aggressive lr
dqn_lr2_steps    = [x[0] for x in dqn_lr2_cb.return_log]
dqn_lr2_surv_log = [x[2] for x in dqn_lr2_cb.return_log]

axes[0].plot(dqn_steps,    dqn_surv_log,    color='steelblue',
             linewidth=2.0, label='DQN lr=1e-4 (default)')
axes[0].plot(dqn_lr2_steps, dqn_lr2_surv_log, color='#aed6f1',
             linewidth=2.0, linestyle='--', label='DQN lr=5e-4')
axes[0].axhline(rand_survival_B, color='gray', linestyle=':',
                linewidth=1.5, label='Random baseline')
axes[0].set_xlabel('Training timesteps'); axes[0].set_ylabel('Survival Rate (%)')
axes[0].set_title('DQN: Learning Rate Sensitivity', fontweight='bold')
axes[0].legend(fontsize=8)

# PPO: default vs aggressive lr
ppo_lr2_steps    = [x[0] for x in ppo_lr2_cb.return_log]
ppo_lr2_surv_log = [x[2] for x in ppo_lr2_cb.return_log]

axes[1].plot(ppo_steps,    ppo_surv_log,    color='darkorange',
             linewidth=2.0, label='PPO lr=3e-4 (default)')
axes[1].plot(ppo_lr2_steps, ppo_lr2_surv_log, color='#fad7a0',
             linewidth=2.0, linestyle='--', label='PPO lr=1e-3')
axes[1].axhline(rand_survival_B, color='gray', linestyle=':',
                linewidth=1.5, label='Random baseline')
axes[1].set_xlabel('Training timesteps'); axes[1].set_ylabel('Survival Rate (%)')
axes[1].set_title('PPO: Learning Rate Sensitivity', fontweight='bold')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configB_lr_sensitivity.png', bbox_inches='tight')
plt.show()

# Summary
print('Learning rate sensitivity summary:')
print(f'  DQN  default (lr=1e-4): {dqn_results["survival_rate"]:.1f}%')
print(f'  DQN  aggressive (lr=5e-4): {dqn_lr2_results["survival_rate"]:.1f}%')
print(f'  PPO  default (lr=3e-4): {ppo_results["survival_rate"]:.1f}%')
print(f'  PPO  aggressive (lr=1e-3): {ppo_lr2_results["survival_rate"]:.1f}%')
print()

# Use best versions going forward
best_dqn = dqn_model if dqn_results['survival_rate'] >= dqn_lr2_results['survival_rate'] else dqn_lr2
best_ppo = ppo_model if ppo_results['survival_rate'] >= ppo_lr2_results['survival_rate'] else ppo_lr2
best_dqn_results = dqn_results if dqn_results['survival_rate'] >= dqn_lr2_results['survival_rate'] else dqn_lr2_results
best_ppo_results = ppo_results if ppo_results['survival_rate'] >= ppo_lr2_results['survival_rate'] else ppo_lr2_results

print(f'Best DQN: survival {best_dqn_results["survival_rate"]:.1f}%')
print(f'Best PPO: survival {best_ppo_results["survival_rate"]:.1f}%')


---
## 12. Robustness Breakdown — Clinical Reality Analysis

This is the centrepiece of Config B. We evaluate both algorithms separately on each episode type produced by the clinical wrappers:

- **Clean episodes**: no wrappers active, baseline performance
- **Noisy episodes**: sensor measurements corrupted (monitor malfunction)
- **Missing feature episodes**: some lab values unavailable
- **Acute event episodes**: sudden irreversible patient deterioration

This analysis directly answers the clinical question: **which algorithm is more robust to the imperfections of real ICU data?**

Note: acute events are irreducible — the patient dies regardless of treatment. A lower survival rate on acute episodes does not reflect poor algorithm performance but rather the fundamental irreversibility of the event.


In [ ]:
# ── Per-episode-type evaluation function ─────────────────────────────────────

def evaluate_by_episode_type(model, n_episodes=2000, seed=SEED):
    """
    Evaluate a model and break down results by episode type:
    clean, noisy, missing features, acute events.

    Returns dict of dicts: {episode_type: {survival_rate, mean_return, n_episodes}}
    """
    np.random.seed(seed)
    env_eval = make_clinical_env()

    results = {
        'clean'  : {'returns': [], 'label': 'Clean'},
        'noisy'  : {'returns': [], 'label': 'Noisy (sensor malfunction)'},
        'missing': {'returns': [], 'label': 'Missing features (lab unavailable)'},
        'acute'  : {'returns': [], 'label': 'Acute event (irreversible)'},
        'overall': {'returns': [], 'label': 'Overall'},
    }

    for _ in range(n_episodes):
        obs, info = env_eval.reset(seed=np.random.randint(100_000))
        ep_noisy   = info.get('noisy_episode', False)
        ep_missing = info.get('missing_features') is not None
        ep_acute   = False
        total_r, done = 0.0, False

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, r, te, tr, info = env_eval.step(int(action))
            total_r += r; done = te or tr
            if info.get('acute_event', False):
                ep_acute = True

        results['overall']['returns'].append(total_r)
        if ep_acute:
            results['acute']['returns'].append(total_r)
        elif ep_noisy:
            results['noisy']['returns'].append(total_r)
        elif ep_missing:
            results['missing']['returns'].append(total_r)
        else:
            results['clean']['returns'].append(total_r)

    env_eval.close()

    # Compute stats
    summary = {}
    for key, val in results.items():
        arr = np.array(val['returns']) if val['returns'] else np.array([0.0])
        summary[key] = {
            'label'        : val['label'],
            'n_episodes'   : len(val['returns']),
            'survival_rate': float(np.mean(arr > 0)) * 100,
            'mean_return'  : float(np.mean(arr)),
            'std_return'   : float(np.std(arr)),
        }
    return summary


print('Evaluating DQN by episode type (2000 episodes)...')
dqn_robustness = evaluate_by_episode_type(best_dqn, n_episodes=2000)

print('Evaluating PPO by episode type (2000 episodes)...')
ppo_robustness = evaluate_by_episode_type(best_ppo, n_episodes=2000)

print('Evaluation complete.')
print()

# Print breakdown
for algo, rob in [('DQN', dqn_robustness), ('PPO', ppo_robustness)]:
    print(f'── {algo} ──')
    print(f'  {"Episode type":<35} {"N":>6} {"Survival %":>12} {"Mean Return":>12}')
    print(f'  {"-"*67}')
    for key in ['overall', 'clean', 'noisy', 'missing', 'acute']:
        r = rob[key]
        print(f'  {r["label"]:<35} {r["n_episodes"]:>6} '
              f'{r["survival_rate"]:>11.1f}% {r["mean_return"]:>12.4f}')
    print()


In [ ]:
# ── Plot: robustness breakdown ────────────────────────────────────────────────

episode_types = ['overall', 'clean', 'noisy', 'missing', 'acute']
type_labels   = ['Overall', 'Clean', 'Noisy\n(sensor)', 'Missing\n(labs)', 'Acute\n(event)']
type_colors   = ['#2c3e50', '#27ae60', '#e67e22', '#8e44ad', '#c0392b']

dqn_survival = [dqn_robustness[t]['survival_rate'] for t in episode_types]
ppo_survival = [ppo_robustness[t]['survival_rate'] for t in episode_types]

# Compute random baseline per episode type
print('Computing random baseline by episode type...')
rand_robustness = evaluate_by_episode_type(
    type('RandomModel', (), {
        'predict': lambda self, obs, deterministic=True:
            (np.random.randint(0, N_ACTIONS), None)
    })(),
    n_episodes=2000
)
rand_survival = [rand_robustness[t]['survival_rate'] for t in episode_types]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: grouped bar chart — DQN vs PPO vs Random per episode type
x     = np.arange(len(episode_types))
width = 0.25

bars_rand = axes[0].bar(x - width, rand_survival, width,
                        label='Random', color='gray', alpha=0.7, edgecolor='white')
bars_dqn  = axes[0].bar(x,          dqn_survival,  width,
                        label='DQN',    color='steelblue', alpha=0.85, edgecolor='white')
bars_ppo  = axes[0].bar(x + width,  ppo_survival,  width,
                        label='PPO',    color='darkorange', alpha=0.85, edgecolor='white')

axes[0].set_xticks(x)
axes[0].set_xticklabels(type_labels, fontsize=9)
axes[0].set_ylabel('Survival Rate (%)')
axes[0].set_title('Robustness Breakdown by Episode Type\n(DQN vs PPO vs Random)',
                  fontweight='bold')
axes[0].set_ylim(0, 100)
axes[0].legend(fontsize=9)
for bars in [bars_rand, bars_dqn, bars_ppo]:
    for bar in bars:
        h = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.5,
                     f'{h:.0f}%', ha='center', va='bottom', fontsize=7)

# Right: performance drop from clean to each failure mode
dqn_clean = dqn_robustness['clean']['survival_rate']
ppo_clean = ppo_robustness['clean']['survival_rate']

failure_types  = ['noisy', 'missing', 'acute']
failure_labels = ['Noisy', 'Missing', 'Acute']

dqn_drops = [dqn_clean - dqn_robustness[t]['survival_rate'] for t in failure_types]
ppo_drops = [ppo_clean - ppo_robustness[t]['survival_rate'] for t in failure_types]

x2    = np.arange(len(failure_types))
width2 = 0.35

axes[1].bar(x2 - width2/2, dqn_drops, width2,
            label='DQN', color='steelblue', alpha=0.85, edgecolor='white')
axes[1].bar(x2 + width2/2, ppo_drops, width2,
            label='PPO', color='darkorange', alpha=0.85, edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xticks(x2)
axes[1].set_xticklabels(failure_labels)
axes[1].set_ylabel('Survival Rate Drop from Clean (pp)')
axes[1].set_title('Performance Drop Due to Clinical Failures\n(higher bar = more affected)',
                  fontweight='bold')
axes[1].legend(fontsize=9)
for bars, drops in [(axes[1].patches[:3], dqn_drops),
                    (axes[1].patches[3:], ppo_drops)]:
    pass  # labels already on bars

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configB_robustness.png', bbox_inches='tight')
plt.show()


---
## 13. Config B — Final Comparison Table


In [ ]:
# ── Config B results table ────────────────────────────────────────────────────

configB_table = pd.DataFrame([
    {
        'Algorithm'     : 'Random Baseline',
        'Survival %'    : round(rand_survival_B, 1),
        'Mean Return'   : round(float(np.mean(clinical_rand_returns)), 4),
        'Std Return'    : round(float(np.std(clinical_rand_returns)), 4),
        'Clean %'       : round(rand_robustness['clean']['survival_rate'], 1),
        'Noisy %'       : round(rand_robustness['noisy']['survival_rate'], 1),
        'Missing %'     : round(rand_robustness['missing']['survival_rate'], 1),
        'Acute %'       : round(rand_robustness['acute']['survival_rate'], 1),
    },
    {
        'Algorithm'     : 'DQN',
        'Survival %'    : round(best_dqn_results['survival_rate'], 1),
        'Mean Return'   : round(best_dqn_results['mean_return'], 4),
        'Std Return'    : round(best_dqn_results['std_return'], 4),
        'Clean %'       : round(dqn_robustness['clean']['survival_rate'], 1),
        'Noisy %'       : round(dqn_robustness['noisy']['survival_rate'], 1),
        'Missing %'     : round(dqn_robustness['missing']['survival_rate'], 1),
        'Acute %'       : round(dqn_robustness['acute']['survival_rate'], 1),
    },
    {
        'Algorithm'     : 'PPO',
        'Survival %'    : round(best_ppo_results['survival_rate'], 1),
        'Mean Return'   : round(best_ppo_results['mean_return'], 4),
        'Std Return'    : round(best_ppo_results['std_return'], 4),
        'Clean %'       : round(ppo_robustness['clean']['survival_rate'], 1),
        'Noisy %'       : round(ppo_robustness['noisy']['survival_rate'], 1),
        'Missing %'     : round(ppo_robustness['missing']['survival_rate'], 1),
        'Acute %'       : round(ppo_robustness['acute']['survival_rate'], 1),
    },
])

display(configB_table)
configB_table.to_csv(f'{PLOTS_DIR}/configB_results_table.csv', index=False)

print()
print('Config B Random baseline survival rate:', rand_survival_B, '%')
print('DQN improvement over random:',
      round(best_dqn_results['survival_rate'] - rand_survival_B, 1), 'pp')
print('PPO improvement over random:',
      round(best_ppo_results['survival_rate'] - rand_survival_B, 1), 'pp')


---
## 14. Config A vs Config B — Cross-Configuration Comparison

This section answers the central question of the project: **what is the cost of moving from a simple discrete MDP to a continuous, noisy, real-world clinical environment?**

We compare the best model-free algorithm from each configuration and examine three dimensions: raw survival rate, stability, and robustness to clinical failures.


In [ ]:
# ── Config A vs Config B: unified comparison ─────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Left: best survival rates across both configs
comp_labels = [
    'Config A\nRandom',
    'Config A\nPI (ceiling)',
    'Config A\nSARSA (opt)',
    'Config A\nQ-Lrn (opt)',
    'Config B\nRandom',
    'Config B\nDQN',
    'Config B\nPPO',
]
comp_values = [
    survival_rate,
    pi_results['survival_rate'],
    sarsa_opt_results['survival_rate'],
    ql_opt_results['survival_rate'],
    rand_survival_B,
    best_dqn_results['survival_rate'],
    best_ppo_results['survival_rate'],
]
comp_colors = [
    'gray', 'tomato', 'steelblue', 'darkorange',
    'lightgray', '#1a5276', '#784212',
]

bars = axes[0].bar(range(len(comp_labels)), comp_values,
                   color=comp_colors, alpha=0.9, edgecolor='white')
axes[0].axvline(3.5, color='black', linewidth=1.0, linestyle='--', alpha=0.4)
axes[0].set_xticks(range(len(comp_labels)))
axes[0].set_xticklabels(comp_labels, fontsize=7)
axes[0].set_ylabel('Survival Rate (%)')
axes[0].set_title('Config A vs Config B\nSurvival Rate Comparison', fontweight='bold')
axes[0].set_ylim(55, 90)
axes[0].text(1.75, 57, 'Config A', ha='center', fontsize=9,
             color='black', alpha=0.5)
axes[0].text(5.25, 57, 'Config B', ha='center', fontsize=9,
             color='black', alpha=0.5)
for bar, val in zip(bars, comp_values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=7, fontweight='bold')

# Middle: improvement over random baseline per config
improvement_labels = ['SARSA\n(Config A)', 'Q-Learning\n(Config A)',
                      'DQN\n(Config B)', 'PPO\n(Config B)']
improvement_values = [
    sarsa_opt_results['survival_rate'] - survival_rate,
    ql_opt_results['survival_rate']    - survival_rate,
    best_dqn_results['survival_rate']  - rand_survival_B,
    best_ppo_results['survival_rate']  - rand_survival_B,
]
imp_colors = ['steelblue', 'darkorange', '#1a5276', '#784212']

bars2 = axes[1].bar(improvement_labels, improvement_values,
                    color=imp_colors, alpha=0.85, edgecolor='white')
axes[1].axvline(1.5, color='black', linewidth=1.0, linestyle='--', alpha=0.4)
axes[1].set_ylabel('Improvement over random baseline (pp)')
axes[1].set_title('Improvement Over Random Baseline\n(pp above random)', fontweight='bold')
axes[1].text(0.5, 0,  'Config A', ha='center', fontsize=9,
             color='black', alpha=0.5, transform=axes[1].get_xaxis_transform())
axes[1].text(2.5, 0, 'Config B', ha='center', fontsize=9,
             color='black', alpha=0.5, transform=axes[1].get_xaxis_transform())
for bar, val in zip(bars2, improvement_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'+{val:.1f}pp', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Right: stability comparison (std of returns)
std_labels = ['SARSA\n(Config A)', 'Q-Lrn\n(Config A)',
              'DQN\n(Config B)', 'PPO\n(Config B)']
std_values = [
    sarsa_best_dist['std_return'],
    ql_best_dist['std_return'],
    best_dqn_results['std_return'],
    best_ppo_results['std_return'],
]
std_colors = ['steelblue', 'darkorange', '#1a5276', '#784212']

axes[2].bar(std_labels, std_values, color=std_colors, alpha=0.85, edgecolor='white')
axes[2].axvline(1.5, color='black', linewidth=1.0, linestyle='--', alpha=0.4)
axes[2].set_ylabel('Std of episode returns')
axes[2].set_title('Policy Stability\n(lower = more reliable)', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configAB_comparison.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Final cross-config summary ────────────────────────────────────────────────

print('=' * 75)
print('CONFIG A vs CONFIG B — FINAL CROSS-CONFIGURATION SUMMARY')
print('=' * 75)
print(f'{"Algorithm":<28} {"Config":<10} {"Survival %":>12} '
      f'{"Std":>8} {"vs Random":>10}')
print('-' * 75)

cross_rows = [
    ('Random Baseline',    'A', survival_rate,
     float(np.std(rand_returns)), 0.0),
    ('Policy Iteration',   'A', pi_results['survival_rate'],
     pi_dist['std_return'],
     pi_results['survival_rate'] - survival_rate),
    ('SARSA (optimised)',  'A', sarsa_opt_results['survival_rate'],
     sarsa_best_dist['std_return'],
     sarsa_opt_results['survival_rate'] - survival_rate),
    ('Q-Learning (opt)',   'A', ql_opt_results['survival_rate'],
     ql_best_dist['std_return'],
     ql_opt_results['survival_rate'] - survival_rate),
    ('Random Baseline',    'B', rand_survival_B,
     float(np.std(clinical_rand_returns)), 0.0),
    ('DQN',                'B', best_dqn_results['survival_rate'],
     best_dqn_results['std_return'],
     best_dqn_results['survival_rate'] - rand_survival_B),
    ('PPO',                'B', best_ppo_results['survival_rate'],
     best_ppo_results['std_return'],
     best_ppo_results['survival_rate'] - rand_survival_B),
]

for name, cfg, sr, std, imp in cross_rows:
    print(f'{name:<28} {"Config "+cfg:<10} {sr:>11.1f}% '
          f'{std:>8.4f} {imp:>+9.1f}pp')

print('=' * 75)
print()
print('Key findings:')
print('1. Both configs show meaningful improvement over their random baselines.')
print('2. Config A benefits from a known model (PI ceiling); Config B does not.')
print('3. The wrappers in Config B create a harder problem — lower random baseline.')
print('4. Deep RL (DQN/PPO) handles continuous observations that tabular')
print('   methods cannot, at the cost of longer training and less interpretability.')
print('5. Robustness to clinical failures (noise, missing data, acute events)')
print('   is the key dimension that Config B adds beyond Config A.')
